# 🎙️ VoiceBatch Studio v2.0.1 - Multilingual & Silence Remover
इसमें 26 भाषाओं का सपोर्ट, Silence Remover और भाषा सुधार (Language Fix) शामिल है।

In [ ]:
# @title 📥 Step 1: इंस्टॉलेशन
print("⏳ सिस्टम तैयार हो रहा है...")
!pip install -q gradio edge-tts librosa soundfile torchcodec coqui-tts
print("✅ सेटअप पूरा हुआ!")

In [ ]:
# @title 💤 Step 2: Anti-Sleep Mode
from IPython.display import display, Javascript
display(Javascript('''
    function ClickConnect(){ document.querySelector("colab-connect-button").click() }
    setInterval(ClickConnect, 60000)
'''))
print("🚀 Anti-Sleep सक्रिय है।")

In [ ]:
# @title 🚀 Step 3: app.py (Multilingual + Silence Remover)
import os

app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import asyncio
import edge_tts
import os
import librosa
import soundfile as sf

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'📥 Loading XTTS v2 on {device}...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def process_audio(audio_path, speed, pitch, remove_silence):
    y, sr = librosa.load(audio_path)
    
    # 1. Silence Part Remover (खामोशी हटाने वाला टूल)
    if remove_silence:
        y, _ = librosa.effects.trim(y, top_db=25)
    
    # 2. Speed Control
    if speed != 1.0:
        y = librosa.effects.time_stretch(y, rate=speed)
    
    # 3. Pitch Control
    if pitch != 0:
        y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
        
    final_path = "final_output.wav"
    sf.write(final_path, y, sr)
    return final_path

async def fast_tts(text, voice, speed, pitch):
    output = 'fast_voice.mp3'
    rate = f"{speed:+}%"
    p = f"{pitch:+}Hz"
    communicate = edge_tts.Communicate(text, voice, rate=rate, pitch=p)
    await communicate.save(output)
    return output

def clone_voice(text, audio_sample, speed_val, pitch_val, lang_code, silence_check):
    if audio_sample is None: return None
    output_path = 'temp_clone.wav'
    # भाषा को यहाँ कड़ाई से लॉक किया गया है (Language Fix)
    tts.tts_to_file(text=text, speaker_wav=audio_sample, language=lang_code, file_path=output_path)
    return process_audio(output_path, speed_val, pitch_val, silence_check)

LANGUAGES = {
    'Hindi': 'hi', 'English': 'en', 'Spanish': 'es', 'French': 'fr', 'German': 'de', 
    'Italian': 'it', 'Portuguese': 'pt', 'Polish': 'pl', 'Turkish': 'tr', 'Russian': 'ru', 
    'Dutch': 'nl', 'Czech': 'cs', 'Arabic': 'ar', 'Chinese': 'zh-cn', 'Japanese': 'ja', 
    'Korean': 'ko', 'Hungarian': 'hu', 'Indonesian': 'id', 'Marathi': 'mr', 'Tamil': 'ta', 
    'Telugu': 'te', 'Kannada': 'kn', 'Gujarati': 'gu', 'Bengali': 'bn', 'Punjabi': 'pa'
}

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.0.1')
    
    with gr.Tabs():
        with gr.TabItem('🧬 Realistic Voice Cloning'):
            with gr.Row():
                with gr.Column():
                    input_text = gr.Textbox(label='टेक्स्ट लिखें', lines=5)
                    sample = gr.Audio(label='वॉइस सैंपल अपलोड करें', type='filepath')
                    lang_drop = gr.Dropdown(choices=list(LANGUAGES.keys()), label="अपनी भाषा चुनें", value='Hindi')
                    with gr.Row():
                        speed_slider = gr.Slider(0.5, 2.0, 1.0, step=0.1, label="Speed")
                        pitch_slider = gr.Slider(-10, 10, 0, step=1, label="Pitch")
                    silence_remover = gr.Checkbox(label="Silence Part Remover", value=True)
                    btn_clone = gr.Button('Realistic Voice Generate 🚀', variant='primary')
                with gr.Column():
                    output_clone = gr.Audio(label='तैयार आवाज़')
            
            btn_clone.click(clone_voice, [input_text, sample, speed_slider, pitch_slider, gr.State(LANGUAGES), silence_remover], output_clone, 
                             preprocess=lambda t, s, sp, pi, d, sil: (t, s, sp, pi, d[lang_drop.value], sil))
            
        with gr.TabItem('⚡ Fast Standard TTS'):
            with gr.Row():
                with gr.Column():
                    t_text = gr.Textbox(label='टेक्स्ट', lines=5)
                    v_drop = gr.Dropdown(choices=['hi-IN-MadhurNeural', 'hi-IN-SwaraNeural'], label='आवाज़', value='hi-IN-MadhurNeural')
                    spd = gr.Slider(-50, 50, 0, label="Speed %")
                    ptc = gr.Slider(-20, 20, 0, label="Pitch")
                    btn_fast = gr.Button('Quick Generate⚡')
                with gr.Column():
                    output_fast = gr.Audio(label='आउटपुट')
            btn_fast.click(lambda t, v, s, p: asyncio.run(fast_tts(t, v, s, p)), [t_text, v_drop, spd, ptc], output_fast)

demo.launch(share=True)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print("✅ v2.0.1 अपडेट हो गया है। अब भाषा नहीं बदलेगी और सन्नाटा भी हट जाएगा।")
!python app.py
      